In [1]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN").strip()
except ImportError:
    if "KAGGLE_API_TOKEN" not in os.environ:
        from getpass import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass("KAGGLE_API_TOKEN: ").strip()


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [4]:
import kagglehub

path = kagglehub.competition_download('multiple-object-video-detection-hw-2-ucu-ai-2026')

print("Path to competition files:", path)

Path to competition files: /Users/ruzhylopv/.cache/kagglehub/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026


# Data Understanding

# Data Preprocessing

# Two-stage Anchor-based

# One-stage Anchor-based


# One-stage Anchor-free

# Query-based


# Custom Pipeline

# Visualizaiton (Bonus)

# Submission

In [ ]:
# ============================================================
# Work in progress, needs cleaning
# ============================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------- Config ----------------
SAMPLE_SUB_PATH = "/kaggle/input/<competition>/sample_submission.csv"  # <-- edit
TEST_IMG_DIR    = "/kaggle/input/<competition>/test/images"            # <-- edit
OUTPUT_PATH     = "submission.csv"

MAX_DETS        = 500      # only the first 500 per image are scored
MIN_SCORE       = 0.001    # drop very low-confidence boxes (keep low for mAP)
ROUND_COORDS    = False    # True -> integer pixel coords like the sample
COORD_DECIMALS  = 2
SCORE_DECIMALS  = 4


# ---------------- Core helpers ----------------
def format_prediction_string(classes, scores, boxes_xywh):
    """
    classes:    (N,) int class ids
    scores:     (N,) float confidences
    boxes_xywh: (N, 4) [x_topleft, y_topleft, width, height] in pixels
    Returns the space-separated PredictionString ('' when there are no detections).
    """
    classes = np.asarray(classes).reshape(-1)
    scores = np.asarray(scores, dtype=float).reshape(-1)
    boxes = np.asarray(boxes_xywh, dtype=float).reshape(-1, 4)

    if len(scores) == 0:
        return ""

    # Filter invalid / low-score boxes
    keep = (scores >= MIN_SCORE) & (boxes[:, 2] > 0) & (boxes[:, 3] > 0)
    classes, scores, boxes = classes[keep], scores[keep], boxes[keep]

    # Keep the top MAX_DETS by score
    order = np.argsort(-scores)[:MAX_DETS]
    classes, scores, boxes = classes[order], scores[order], boxes[order]

    parts = []
    for c, s, (x, y, w, h) in zip(classes, scores, boxes):
        if ROUND_COORDS:
            coords = f"{int(round(x))} {int(round(y))} {int(round(w))} {int(round(h))}"
        else:
            d = COORD_DECIMALS
            coords = f"{x:.{d}f} {y:.{d}f} {w:.{d}f} {h:.{d}f}"
        parts.append(f"{int(c)} {s:.{SCORE_DECIMALS}f} {coords}")
    return " ".join(parts)


def xyxy_to_xywh(boxes_xyxy, img_w=None, img_h=None):
    """Convert [x1, y1, x2, y2] -> [x, y, w, h] (top-left), optionally clipping to the image."""
    b = np.asarray(boxes_xyxy, dtype=float).reshape(-1, 4).copy()
    if img_w is not None and img_h is not None:
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, img_w)
        b[:, [1, 3]] = b[:, [1, 3]].clip(0, img_h)
    return np.stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)


def build_submission(predictions, sample_sub_path=SAMPLE_SUB_PATH, output_path=OUTPUT_PATH):
    """
    predictions: dict {image_id: (classes, scores, boxes_xywh)}
    Uses sample_submission.csv for the image list and order; images with no
    prediction get an empty PredictionString.
    """
    sample = pd.read_csv(sample_sub_path)
    sample_ids = sample["image_id"].astype(str).tolist()

    rows = []
    for img_id in sample_ids:
        if img_id in predictions:
            c, s, b = predictions[img_id]
            rows.append(format_prediction_string(c, s, b))
        else:
            rows.append("")

    sub = pd.DataFrame({"image_id": sample_ids, "PredictionString": rows})

    # Sanity checks
    extra = set(predictions) - set(sample_ids)
    if extra:
        print(f"[warn] {len(extra)} predicted image_ids are not in sample_submission "
              f"(e.g. {list(extra)[:3]}); they were ignored.")
    assert len(sub) == len(sample), "Row count mismatch with sample_submission"
    assert sub["image_id"].is_unique, "Duplicate image_ids in submission"

    # Empty strings must be written as empty, not NaN
    sub["PredictionString"] = sub["PredictionString"].fillna("")
    sub.to_csv(output_path, index=False)

    n_empty = (sub["PredictionString"] == "").sum()
    n_dets = sub["PredictionString"].str.split().str.len().fillna(0).sum() // 6
    print(f"Saved {output_path}: {len(sub)} rows | {n_empty} empty | {int(n_dets)} total detections")
    return sub


# ---------------- Optional: Ultralytics YOLO inference -> predictions ----------------
def predict_with_ultralytics(model, img_dir=TEST_IMG_DIR, imgsz=640, conf=MIN_SCORE,
                             iou=0.6, batch=16, augment=False, device=None):
    """Run a YOLO model over img_dir and return a predictions dict for build_submission."""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    paths = sorted(str(p) for p in Path(img_dir).iterdir() if p.suffix.lower() in exts)

    predictions = {}
    for i in range(0, len(paths), batch):
        results = model.predict(
            paths[i:i + batch], imgsz=imgsz, conf=conf, iou=iou,
            max_det=MAX_DETS, augment=augment, device=device, verbose=False,
        )
        for r in results:
            img_id = Path(r.path).stem
            h, w = r.orig_shape
            if r.boxes is None or len(r.boxes) == 0:
                predictions[img_id] = ([], [], np.zeros((0, 4)))
                continue
            xyxy = r.boxes.xyxy.cpu().numpy()          # already in original-image pixels
            predictions[img_id] = (
                r.boxes.cls.cpu().numpy().astype(int),
                r.boxes.conf.cpu().numpy(),
                xyxy_to_xywh(xyxy, w, h),
            )
        print(f"\rInference: {min(i + batch, len(paths))}/{len(paths)}", end="")
    print()
    return predictions


# ---------------- Run ----------------
# from ultralytics import YOLO
# model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")
# predictions = predict_with_ultralytics(model, imgsz=1280)
#
# Or supply your own dict:
# predictions = {"clip_001_0000001": ([2, 0], [0.91, 0.55], [[1366, 580, 26, 44], [809, 625, 29, 66]])}

submission = build_submission(predictions)
submission.head(10)

In [ ]:
!kaggle competitions submit -c multiple-object-video-detection-hw-2-ucu-ai-2026 -f submission.csv -m "Proba 1"